# Régression logistique : exploration

Ce notebook teste la régression logistique avec les mêmes règles que les autres modèles : découpage commun, validation croisée stratifiée à 5 plis sur le jeu d'entraînement, PR-AUC comme métrique principale et seuil de décision à 0,5 (on optimisera celui du modèle qu'on retiendra à la fin). Le jeu de test n'est pas utilisé, il reste de côté pour l'évaluation finale.

L'idée c'est de partir de la version brute, puis d'ajouter une par une les variables qui viennent de l'EDA pour voir ce que chacune apporte, et enfin de régler la régularisation. Chaque essai est enregistré dans MLflow.

## Importations et pré-réglages

In [ ]:
import mlflow
import pandas as pd
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from base_commune.outils import (
    RANDOM_STATE,
    charger_donnees,
    configurer_mlflow,
    decouper_donnees,
    evaluer_modele,
    exporter_runs,
    tags_reproductibilite,
)

AUTEUR = "david"
CREER_LES_RUNS = False  # True pour relancer les runs

In [ ]:
df_brut = charger_donnees()
X_train, X_test, y_train, y_test = decouper_donnees(df_brut)
print(X_train.shape, X_test.shape)

## Pipeline

Variables ajoutées à partir de l'EDA : NumOfProducts en catégories, l'âge en tranches ou au carré, l'interaction âge × inactif et une indicatrice solde nul.

In [ ]:
BORNES_AGE = [18, 28, 38, 48, 58, 68, 78, 88, 98]


def ajouter_variables(X, options=()):
    X = X.copy()
    if "age_tranches" in options:
        X["Age_tranche"] = pd.cut(X["Age"], bins=BORNES_AGE, right=False).astype(str)
    if "age_carre" in options:
        X["Age2"] = X["Age"] ** 2
    if "interaction" in options:
        X["AgeXInactif"] = X["Age"] * (1 - X["IsActiveMember"])
    if "solde_nul" in options:
        X["SoldeNul"] = (X["Balance"] == 0).astype(int)
    return X


def colonnes(options=()):
    cate = ["Geography", "Gender"]
    num = ["CreditScore", "Tenure", "Balance", "EstimatedSalary", "HasCrCard", "IsActiveMember"]
    num += [] if "produits_cate" in options else ["NumOfProducts"]
    cate += ["NumOfProducts"] if "produits_cate" in options else []
    if "age_tranches" in options:
        cate += ["Age_tranche"]
    else:
        num += ["Age"]
    num += [nom for opt, nom in [("age_carre", "Age2"), ("interaction", "AgeXInactif"),
                                 ("solde_nul", "SoldeNul")] if opt in options]
    return num, cate


def construire_reglog(options=(), C=1.0, l1_ratio=0.0, class_weight=None):
    num, cate = colonnes(options)
    preprocesseur = ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cate", OneHotEncoder(handle_unknown="ignore"), cate),
    ], remainder="drop")
    return Pipeline([
        ("variables", FunctionTransformer(ajouter_variables, kw_args={"options": options})),
        ("preprocesseur", preprocesseur),
        ("reglog", LogisticRegression(C=C, l1_ratio=l1_ratio, class_weight=class_weight,
                                      solver="saga", max_iter=5000,
                                      random_state=RANDOM_STATE)),
    ])

## Suivi MLflow

In [ ]:
configurer_mlflow("churn_bancaire")


def enregistrer_run(modele, nom_run, famille, params, options=(), logger_modele=False,
                    stage="exploration"):
    num, cate = colonnes(options)
    with mlflow.start_run(run_name=nom_run):
        mlflow.set_tags({"model_family": famille, "author": AUTEUR, "stage": stage,
                         **tags_reproductibilite()})
        mlflow.log_params({**params, "variables_ajoutees": ",".join(options) or "aucune",
                           "features": ",".join(num + cate),
                           "cv_folds": 5, "random_state": RANDOM_STATE,
                           "test_size": 0.2, "n_train": len(X_train)})
        metriques = evaluer_modele(modele, X_train, y_train)
        mlflow.log_metrics(metriques)
        if logger_modele:
            modele.fit(X_train, y_train)
            signature = infer_signature(X_train, modele.predict_proba(X_train)[:, 1])
            mlflow.sklearn.log_model(modele, name="model", signature=signature,
                                     input_example=X_train.head(3),
                                     serialization_format="cloudpickle")
    return metriques


def afficher(nom, m):
    print(f"{nom:45s} PR-AUC {m['cv_pr_auc_mean']:.3f} (± {m['cv_pr_auc_std']:.3f})"
          f"  ROC-AUC {m['cv_roc_auc_mean']:.3f}  recall {m['cv_recall_mean']:.3f}")

## Version brute

In [ ]:
if CREER_LES_RUNS:
    m = enregistrer_run(construire_reglog(), "reglog_brute", "logistic_regression",
                        {"C": 1.0, "l1_ratio": 0.0, "class_weight": None})
    afficher("reglog_brute", m)

PR-AUC de 0,469, loin du KNN (0,621) et du Random Forest (0,68). Le modèle ne voit que des effets linéaires, alors que l'âge et le nombre de produits n'en ont pas.

## Ajout des variables de l'EDA

On les ajoute une par une en gardant les précédentes.

In [ ]:
ETAPES = [
    ("reglog_produits_cate", ("produits_cate",)),
    ("reglog_produits_age_tranches", ("produits_cate", "age_tranches")),
    ("reglog_produits_age_carre", ("produits_cate", "age_carre")),
    ("reglog_produits_age_carre_interaction", ("produits_cate", "age_carre", "interaction")),
    ("reglog_produits_age_carre_interaction_solde", ("produits_cate", "age_carre", "interaction",
                                                     "solde_nul")),
]

if CREER_LES_RUNS:
    for nom, options in ETAPES:
        m = enregistrer_run(construire_reglog(options), nom, "logistic_regression",
                            {"C": 1.0, "l1_ratio": 0.0, "class_weight": None}, options)
        afficher(nom, m)

Le plus gros gain vient de NumOfProducts en catégories (0,617). L'âge fait monter à 0,654 en tranches et à 0,653 au carré. On garde l'âge au carré parce qu'il permet de faire l'interaction, qui amène à 0,665. Le solde nul n'ajoute rien une fois le pays dans le modèle.

## Régularisation

On fait varier C, la pénalité (L2 ou L1) et class_weight.

In [ ]:
OPTIONS_RETENUES = ("produits_cate", "age_carre", "interaction", "solde_nul")

if CREER_LES_RUNS:
    for C_ in [0.01, 0.1, 1.0, 10.0]:
        for l1_ratio in [0.0, 1.0]:
            for class_weight in [None, "balanced"]:
                params = {"C": C_, "l1_ratio": l1_ratio, "class_weight": class_weight}
                nom = f"reglog_C{C_}_l1ratio{l1_ratio}_cw{class_weight}"
                m = enregistrer_run(construire_reglog(OPTIONS_RETENUES, **params), nom,
                                    "logistic_regression", params, OPTIONS_RETENUES)
                afficher(nom, m)

Avec C = 0,01 on régularise trop (entre 0,57 et 0,60). À partir de C = 1 tout se vaut, en L1 comme en L2, et les écarts restent plus petits que l'écart-type entre les plis. class_weight="balanced" fait passer le recall de 43 % à 75 % sans changer le PR-AUC, ça revient à baisser le seuil.

On garde les réglages par défaut : PR-AUC de 0,665 et ROC-AUC de 0,844.

## Export des runs

In [ ]:
chemin = exporter_runs("churn_bancaire", AUTEUR)
print(chemin)

## Coefficients du modèle retenu

Les variables numériques sont standardisées, donc les coefficients se comparent entre eux.

In [ ]:
modele_retenu = construire_reglog(OPTIONS_RETENUES, C=1.0, l1_ratio=0.0)
modele_retenu.fit(X_train, y_train)

noms = modele_retenu.named_steps["preprocesseur"].get_feature_names_out()
coefs = pd.Series(modele_retenu.named_steps["reglog"].coef_[0], index=noms)
coefs.sort_values(key=abs, ascending=False).round(3)

Avoir 2 produits fait baisser le risque (-2,70), alors que 3 et 4 produits l'augmentent (+1,36 et +2,51, sur seulement 50 clients pour 4 produits). L'âge et l'âge au carré donnent la courbe en cloche de l'EDA.

Le coefficient positif d'IsActiveMember se compense avec l'interaction : être inactif ne change presque rien vers 25-30 ans, mais multiplie la cote de départ par environ 3 à 45 ans et par plus de 6 à 55 ans.

Les clients allemands et les femmes partent plus. Balance, CreditScore, EstimatedSalary, Tenure et HasCrCard ne pèsent presque rien.

## Conclusion

Avec les variables de l'EDA, la régression logistique arrive à 0,665 de PR-AUC : devant le KNN (0,621), un peu derrière le Random Forest (0,68), et beaucoup plus simple à expliquer.